# pinball_endpoint_2

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import matplotlib.pyplot as plt

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load each agent's stored runs, skipping agents with none stored yet.
AGENTS = {
    "DDQN, buffer 10k": "ddqn_large",
    "DDQN, buffer 500": "ddqn_small",
    "DDQN, buffer 500, 10-step": "ddqn_small_nstep",
    "Endpoint, 100 recent + 400 10-step": "endpoint",
}
results = {agent: load_results(EXPERIMENT, name) for agent, name in AGENTS.items()}
results = {agent: runs for agent, runs in results.items() if runs is not None}

In [ ]:
# Compute each agent's mean return over seeds with a 95% bootstrap CI.
curves = {}
for agent, runs in results.items():
    timesteps, returns = subsample(return_curve(runs.reward, runs.done))
    curves[agent] = (timesteps, mean_ci(returns), len(returns))

In [ ]:
# Style for the learning-curve plot.
STYLE = {
    "figure.figsize": (9, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
    "legend.loc": "lower right",
}
COLORS = {
    "ddqn_large": "black",
    "ddqn_small": "tab:blue",
    "ddqn_small_nstep": "tab:purple",
    "endpoint": "tab:red",
}
BAND_ALPHA = 0.2
LEGEND_LABEL = "{agent} (n={runs})"
TITLE = "DDQN buffers vs. endpoint replay on Pinball easy  (mean ± 95% bootstrap CI)"
XLABEL = "Timestep"
YLABEL = "Return"
YLABEL_KW = {"rotation": 0, "ha": "right", "va": "center", "labelpad": 12}
YLIM = (-1000, 0)

In [ ]:
# Plot each agent's mean return and CI band over time.
figures = {}
if curves:
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots()
        for agent, (timesteps, (mean, low, high), runs) in curves.items():
            color = COLORS[AGENTS[agent]]
            ax.fill_between(timesteps, low, high, color=color, alpha=BAND_ALPHA)
            label = LEGEND_LABEL.format(agent=agent, runs=runs)
            ax.plot(timesteps, mean, color=color, label=label)
        ax.set(title=TITLE, xlabel=XLABEL, ylim=YLIM)
        ax.set_ylabel(YLABEL, **YLABEL_KW)
        ax.legend()
        fig.tight_layout()
    figures["ddqn_vs_endpoint"] = fig
    plt.show()

In [ ]:
# Save every plotted figure as a PDF in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")